In [ ]:
from collections import Counter, defaultdict
import math
import re
import urllib.request
import numpy as np

np.random.seed(42)

# Map target language codes directly to IndicCorpV2 files
LANG_FILE_MAP = {
    "as": "as.txt",
    "bn": "bn.txt",
    "brx": "bd.txt",
    "doi": "dg.txt",
    "gom": "gom.txt",
    "gu": "gu.txt",
    "hi": "hi-1.txt",
    "kn": "kn.txt",
    "ks": "ks.txt",
    "mai": "mai.txt",
    "ml": "ml.txt",
    "mni": "mni.txt",
    "mr": "mr.txt",
    "ne": "ne.txt",
    "or": "or.txt",
    "pa": "pa.txt",
    "sa": "sa.txt",
    "sat": "sat.txt",
    "sd": "sd.txt",
    "te": "te.txt",
    "ta": "ta.txt",
    "ur": "ur.txt",
    "en": "en.txt",
}


def sentence_tokenize(text):
    """Splits text paragraphs into individual clean sentences."""
    sentences = re.split(r"(?<=[।॥.!?])\s+", str(text))
    return [s.strip() for s in sentences if len(s.strip()) > 15]


def fetch_dataset_partial_stream(samples_per_lang=200):
    """Fetch dataset using direct byte-range HTTP streaming without requiring HF tokens."""
    print("--- 1. Partial Byte-Streaming Sentences from IndicCorpV2 ---")
    data_by_lang = defaultdict(list)

    for lang_code, filename in LANG_FILE_MAP.items():
        url = f"https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/{filename}"
        collected = []

        try:
            # Request only the first 1MB chunk using HTTP Range headers
            req = urllib.request.Request(
                url,
                headers={
                    "User-Agent": "Mozilla/5.0",
                    "Range": "bytes=0-1048576",
                },
            )

            with urllib.request.urlopen(req) as response:
                chunk = response.read().decode("utf-8", errors="ignore")
                lines = chunk.splitlines()

                for line in lines[:-1]:  # Drop partial last line
                    line_str = line.strip()
                    if not line_str:
                        continue

                    for sent in sentence_tokenize(line_str):
                        collected.append(sent)
                        if len(collected) >= samples_per_lang:
                            break

                    if len(collected) >= samples_per_lang:
                        break

            data_by_lang[lang_code] = collected[:samples_per_lang]
            print(f" -> [{lang_code}] Loaded {len(collected)} sentences.")

        except Exception as e:
            print(f" -> [{lang_code}] Skipping due to stream error: {e}")

    return data_by_lang


def stratified_split(data_by_lang, train_ratio=0.8, val_ratio=0.2):
    train_x, train_y = [], []
    val_x, val_y = [], []

    label_map = {lang: idx for idx, lang in enumerate(data_by_lang.keys())}

    for lang, sentences in data_by_lang.items():
        if len(sentences) == 0:
            continue
        np.random.shuffle(sentences)
        n = len(sentences)
        n_train = int(n * train_ratio)

        label = label_map[lang]

        train_x.extend(sentences[:n_train])
        train_y.extend([label] * n_train)

        val_x.extend(sentences[n_train:])
        val_y.extend([label] * (n - n_train))

    return (
        (train_x, np.array(train_y, dtype=np.int64)),
        (val_x, np.array(val_y, dtype=np.int64)),
        label_map,
    )


# =====================================================================
# 2. FEATURE EXTRACTOR IMPLEMENTING ASSIGNMENT 2 (Q1 & Q2)
# =====================================================================


class Assignment2TFIDFVectorizer:

    def __init__(self, max_features=5000):
        self.max_features = max_features
        self.vocabulary_ = {}
        self.doc_counts = None
        self.total_docs = 0

    def fit(self, raw_documents):
        self.total_docs = len(raw_documents)
        df = defaultdict(int)

        for doc in raw_documents:
            tokens = set(doc.lower().split())
            for token in tokens:
                df[token] += 1

        # Select top most frequent words for vocabulary
        sorted_features = sorted(df.items(), key=lambda x: x[1], reverse=True)[
            : self.max_features
        ]
        self.vocabulary_ = {
            feature: idx for idx, (feature, _) in enumerate(sorted_features)
        }

        # Store Document Frequencies for IDF computations
        self.doc_counts = np.zeros(len(self.vocabulary_))
        for feature, idx in self.vocabulary_.items():
            self.doc_counts[idx] = df[feature]

        return self

    # --- QUESTION 1: IMPLEMENTING TF VARIATIONS ---
    def compute_tf_matrices(self, raw_documents):
        num_docs = len(raw_documents)
        num_words = len(self.vocabulary_)

        # Unnormalized TF (Raw Count)
        tf_unnorm = np.zeros((num_docs, num_words), dtype=np.float32)
        # Q1a: Normalized based on total number of words in sentence (word count/total words)
        tf_norm_len = np.zeros((num_docs, num_words), dtype=np.float32)
        # Q1b: Normalized based on frequency of most frequent word in sentence (word count/max word count)
        tf_norm_max = np.zeros((num_docs, num_words), dtype=np.float32)

        for doc_idx, doc in enumerate(raw_documents):
            tokens = doc.lower().split()
            counts = Counter(tokens)
            total_words = len(tokens) if len(tokens) > 0 else 1.0
            max_freq = max(counts.values()) if counts else 1.0

            for token, count in counts.items():
                if token in self.vocabulary_:
                    col_idx = self.vocabulary_[token]
                    tf_unnorm[doc_idx, col_idx] = count
                    tf_norm_len[doc_idx, col_idx] = count / total_words
                    tf_norm_max[doc_idx, col_idx] = count / max_freq

        return tf_unnorm, tf_norm_len, tf_norm_max

    # --- QUESTION 2: IMPLEMENTING IDF VARIATIONS ---
    def compute_idf_vectors(self):
        # Unnormalized Standard IDF: log(N / DF)
        idf_unnorm = np.log((self.total_docs + 1) / (self.doc_counts + 1)) + 1.0

        # Q2: Normalized IDF based on most frequent word across sentences (Max DF) (scaling rarity using most common word in entire dataset)
        max_df_across_sentences = np.max(self.doc_counts)
        idf_norm = (
            np.log((max_df_across_sentences + 1) / (self.doc_counts + 1)) + 1.0
        )

        return idf_unnorm, idf_norm


# =====================================================================
# 3. CUSTOM LOGISTIC REGRESSION FROM SCRATCH
# =====================================================================


class CustomLogisticRegression:

    def __init__(self, lr=0.5, epochs=50, batch_size=128):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = None

    def _softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y, num_classes):
        num_samples, num_features = X.shape

        self.weights = np.zeros((num_features, num_classes), dtype=np.float32)
        self.bias = np.zeros((1, num_classes), dtype=np.float32)

        y_encoded = np.zeros((num_samples, num_classes), dtype=np.float32)
        y_encoded[np.arange(num_samples), y.astype(int)] = 1.0

        for epoch in range(self.epochs):
            indices = np.arange(num_samples)
            np.random.shuffle(indices)

            for start in range(0, num_samples, self.batch_size):
                end = start + self.batch_size
                b_idx = indices[start:end]

                X_b, y_b = X[b_idx], y_encoded[b_idx]

                logits = np.dot(X_b, self.weights) + self.bias
                probs = self._softmax(logits)

                error = probs - y_b
                dw = np.dot(X_b.T, error) / len(b_idx)
                db = np.sum(error, axis=0, keepdims=True) / len(b_idx)

                self.weights -= self.lr * dw
                self.bias -= self.lr * db

    def predict(self, X):
        logits = np.dot(X, self.weights) + self.bias
        return np.argmax(self._softmax(logits), axis=1)


# =====================================================================
# 4. CUSTOM ACCURACY EVALUATION METRIC
# =====================================================================


def compute_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


# =====================================================================
# MAIN EXECUTION PIPELINE (QUESTION 3 EVALUATION)
# =====================================================================

if __name__ == "__main__":
    # 1. Stream text data
    data_by_lang = fetch_dataset_partial_stream(samples_per_lang=200)
    (train_x, train_y), (val_x, val_y), label_map = stratified_split(
        data_by_lang
    )
    num_classes = len(label_map)

    print(
        f"\nDataset Ready -> Train sentences: {len(train_x)}, Val sentences: {len(val_x)}"
    )

    # 2. Extract TF & IDF matrices
    print("\n--- 2. Extracting Normalized TF and IDF Variants ---")
    vectorizer = Assignment2TFIDFVectorizer(max_features=3000)
    vectorizer.fit(train_x)

    # Train Matrices
    tf_unnorm_train, tf_norm_len_train, tf_norm_max_train = (
        vectorizer.compute_tf_matrices(train_x)
    )
    # Val Matrices
    tf_unnorm_val, tf_norm_len_val, tf_norm_max_val = (
        vectorizer.compute_tf_matrices(val_x)
    )

    # IDF Vectors
    idf_unnorm, idf_norm = vectorizer.compute_idf_vectors()

    # 3. Create all 6 feature combination pairs (Question 3)
    train_combinations = {
        "1. Unnorm TF + Unnorm IDF": tf_unnorm_train * idf_unnorm,
        "2. Unnorm TF + Norm IDF": tf_unnorm_train * idf_norm,
        "3. Sentence-Len TF + Unnorm IDF": tf_norm_len_train * idf_unnorm, #normalised sentence length tf * unnormalised idf
        "4. Sentence-Len TF + Norm IDF": tf_norm_len_train * idf_norm, #normalised sentence length tf * normalised idf
        "5. Max-Freq TF + Unnorm IDF": tf_norm_max_train * idf_unnorm, #normalised max frequency tf * unnormalised idf
        "6. Max-Freq TF + Norm IDF": tf_norm_max_train * idf_norm, #normalised max frequency tf * normalised idf
    }

    val_combinations = {
        "1. Unnorm TF + Unnorm IDF": tf_unnorm_val * idf_unnorm,
        "2. Unnorm TF + Norm IDF": tf_unnorm_val * idf_norm,
        "3. Sentence-Len TF + Unnorm IDF": tf_norm_len_val * idf_unnorm,
        "4. Sentence-Len TF + Norm IDF": tf_norm_len_val * idf_norm,
        "5. Max-Freq TF + Unnorm IDF": tf_norm_max_val * idf_unnorm,
        "6. Max-Freq TF + Norm IDF": tf_norm_max_val * idf_norm,
    }

    # 4. Train and Compare Logistic Regression Models across combinations
    print("\n" + "=" * 68)
    print("--- 3. QUESTION 3: EVALUATING ALL 6 COMBINATIONS ---")
    print("=" * 68)
    print(f"{'TF-IDF Variant Combination':<40} | {'Val Accuracy':<10}")
    print("-" * 68)

    for combo_name in train_combinations.keys():
        X_train = train_combinations[combo_name]
        X_val = val_combinations[combo_name]

        # Train Logistic Regression
        clf = CustomLogisticRegression(lr=1.0, epochs=60, batch_size=128)
        clf.fit(X_train, train_y, num_classes)

        # Evaluate performance
        preds = clf.predict(X_val)
        acc = compute_accuracy(val_y, preds)

        print(f"{combo_name:<40} | {acc * 100:.2f}%")

    print("=" * 68)

--- 1. Partial Byte-Streaming Sentences from IndicCorpV2 ---
 -> [as] Loaded 200 sentences.
 -> [bn] Loaded 200 sentences.
 -> [brx] Loaded 200 sentences.
 -> [doi] Loaded 200 sentences.
 -> [gom] Loaded 200 sentences.
 -> [gu] Loaded 200 sentences.
 -> [hi] Loaded 200 sentences.
 -> [kn] Loaded 200 sentences.
 -> [ks] Loaded 200 sentences.
 -> [mai] Loaded 200 sentences.
 -> [ml] Loaded 200 sentences.
 -> [mni] Loaded 200 sentences.
 -> [mr] Loaded 200 sentences.
 -> [ne] Loaded 200 sentences.
 -> [or] Loaded 200 sentences.
 -> [pa] Loaded 200 sentences.
 -> [sa] Loaded 200 sentences.
 -> [sat] Loaded 200 sentences.
 -> [sd] Loaded 200 sentences.
 -> [te] Loaded 200 sentences.
 -> [ta] Loaded 200 sentences.
 -> [ur] Loaded 200 sentences.
 -> [en] Loaded 200 sentences.

Dataset Ready -> Train sentences: 3680, Val sentences: 920

--- 2. Extracting Normalized TF and IDF Variants ---

--- 3. QUESTION 3: EVALUATING ALL 6 COMBINATIONS ---
TF-IDF Variant Combination               | Val Accur